# YOLO instance segmentation experiment

This notebook is a clean experimental wrapper for evaluating YOLO-based instance segmentation against the current U-Net baseline. It keeps the existing U-Net training pipeline untouched and isolates the YOLO experiment in a separate experimental path.

Core goals:
- reproduce the current 565/71/71 split exactly
- convert COCO annotations to YOLO segmentation format
- train YOLO26n-seg initially
- evaluate validation and local test using PQ/SQ/RQ
- compare against the current U-Net baseline

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

warnings.filterwarnings('ignore')

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

raw_data_dir = project_root / 'data' / 'raw' / 'MAGFiLO_1.0_Kaggle_2026'
train_images_dir = raw_data_dir / 'train' / 'train_images'
annotation_path = raw_data_dir / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
experiment_root = project_root / 'artifacts' / 'experiments' / 'yolo_instance_segmentation'
experiment_root.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Dataset annotations: {annotation_path}')
print(f'Train images dir: {train_images_dir}')

Project root: c:\Users\Darío\Desktop\KAGGLE\kaggle_solar_filament_segmentation
Dataset annotations: c:\Users\Darío\Desktop\KAGGLE\kaggle_solar_filament_segmentation\data\raw\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train.json
Train images dir: c:\Users\Darío\Desktop\KAGGLE\kaggle_solar_filament_segmentation\data\raw\MAGFiLO_1.0_Kaggle_2026\train\train_images


In [2]:
with open(annotation_path, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

print('keys:', list(annotations.keys()))
print('n_images:', len(annotations['images']))
print('n_annotations:', len(annotations['annotations']))
print('first_image_keys:', list(annotations['images'][0].keys()))
print('first_image_sample:', annotations['images'][0])
print('first_annotation_keys:', list(annotations['annotations'][0].keys()))
print('first_annotation_sample:', annotations['annotations'][0])

seg_counts = {'list': 0, 'dict': 0, 'other': 0}
multi_polygon_count = 0
for ann in annotations['annotations']:
    seg = ann.get('segmentation')
    if isinstance(seg, list):
        if len(seg) > 1:
            multi_polygon_count += 1
        for item in seg:
            if isinstance(item, list):
                seg_counts['list'] += 1
            elif isinstance(item, dict):
                seg_counts['dict'] += 1
            else:
                seg_counts['other'] += 1
    else:
        seg_counts['other'] += 1

print('segmentation_format_summary:', seg_counts)
print('annotations_with_multiple_polygons:', multi_polygon_count)
print('sample_image_lookup:', {img['id']: img['file_name'] for img in annotations['images'][:3]})

keys: ['info', 'licenses', 'categories', 'images', 'annotations']
n_images: 1154
n_annotations: 8199
first_image_keys: ['license', 'file_name', 'url', 'height', 'width', 'date_captured', 'id']
first_image_sample: {'license': 1, 'file_name': '20140609195854Bh.jpeg', 'url': 'https://gong2.nso.edu/HA/hag/201406/20140609/20140609195854Bh.jpg', 'height': 2048, 'width': 2048, 'date_captured': '2014-06-09 19:58:54', 'id': '040301-20140609195854Bh'}
first_annotation_keys: ['segmentation', 'area', 'iscrowd', 'spine', 'image_id', 'bbox', 'category_id', 'id']
first_annotation_sample: {'segmentation': [[480.9382, 850.955, 480.307, 852.0535, 478.2989, 852.451, 476.1202, 851.3087, 471.8813, 845.6188, 469.2107, 844.1978, 460.0, 845.0, 459.0, 844.0, 455.9884, 843.2471, 452.2964, 839.9584, 450.6544, 839.7575, 449.7991, 838.6671, 450.223, 837.6258, 447.5581, 834.6278, 445.4449, 829.1312, 442.9577, 825.6077, 441.8277, 824.4777, 438.6437, 823.5973, 439.0, 821.0, 436.0, 818.0, 435.1557, 815.4672, 433.9753,

In [3]:
from src.processing.masks import build_masks
from src.splitting.split import split_masks
from src.utils.data import load_annotations, remove_duplicate_annotations

annotations_clean = remove_duplicate_annotations(load_annotations(annotation_path))
masks_by_filename = build_masks(annotations_clean)
train_masks, val_masks, test_masks = split_masks(
    masks=masks_by_filename,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    random_state=42,
)

print('train_size:', len(train_masks))
print('val_size:', len(val_masks))
print('test_size:', len(test_masks))
print('sum_size:', len(train_masks) + len(val_masks) + len(test_masks))

train_size: 565
val_size: 71
test_size: 71
sum_size: 707


## YOLO configuration and environment

The next cell checks the installed Ultralytics library and configures the training setup. The objective is to verify the exact API before fixing the experiment parameters.

In [4]:
import importlib

try:
    import ultralytics
    from ultralytics import YOLO
    print('ultralytics_version:', getattr(ultralytics, '__version__', 'unknown'))
    print('YOLO_type:', YOLO)
    print('has_train:', hasattr(YOLO, 'train'))
    print('has_predict:', hasattr(YOLO, 'predict'))
    print('has_segment_model:', hasattr(YOLO, 'segment'))
except Exception as exc:
    print('Ultralytics is not installed or import failed:', repr(exc))
    print('Install with: pip install ultralytics')

ultralytics_version: 8.4.161
YOLO_type: <class 'ultralytics.models.yolo.model.YOLO'>
has_train: True
has_predict: True
has_segment_model: False


In [5]:
from src.experimental.yolo.dataset import prepare_yolo_segmentation_dataset

dataset_info = prepare_yolo_segmentation_dataset(
    raw_data_dir=raw_data_dir,
    output_dir=experiment_root,
    random_state=42,
    class_name='filament',
)

print(dataset_info)

{'dataset_root': WindowsPath('c:/Users/Darío/Desktop/KAGGLE/kaggle_solar_filament_segmentation/artifacts/experiments/yolo_instance_segmentation/yolo_segmentation'), 'train': WindowsPath('c:/Users/Darío/Desktop/KAGGLE/kaggle_solar_filament_segmentation/artifacts/experiments/yolo_instance_segmentation/yolo_segmentation/train'), 'val': WindowsPath('c:/Users/Darío/Desktop/KAGGLE/kaggle_solar_filament_segmentation/artifacts/experiments/yolo_instance_segmentation/yolo_segmentation/val'), 'test': WindowsPath('c:/Users/Darío/Desktop/KAGGLE/kaggle_solar_filament_segmentation/artifacts/experiments/yolo_instance_segmentation/yolo_segmentation/test'), 'yaml': WindowsPath('c:/Users/Darío/Desktop/KAGGLE/kaggle_solar_filament_segmentation/artifacts/experiments/yolo_instance_segmentation/yolo_segmentation/dataset.yaml'), 'class_name': 'filament'}


## Experiment configuration

The default config below is intentionally cautious and matches the requirement: start from YOLO26n-seg, use the same split as the U-Net pipeline, and keep the evaluation logic explicit.

In [6]:
EXPERIMENT = {
    'model_name': 'yolo26n-seg.pt',
    'imgsz': 1024,
    'epochs': 30,
    'batch': 4,
    'pretrained': True,
    'seed': 42,
    'confidence_threshold': 0.25,
    'iou_threshold': 0.5,
    'split': {'train': 565, 'val': 71, 'test': 71},
    'default_thresholds': [0.15, 0.25, 0.35, 0.5],
}

print(EXPERIMENT)

{'model_name': 'yolo26n-seg.pt', 'imgsz': 1024, 'epochs': 30, 'batch': 4, 'pretrained': True, 'seed': 42, 'confidence_threshold': 0.25, 'iou_threshold': 0.5, 'split': {'train': 565, 'val': 71, 'test': 71}, 'default_thresholds': [0.15, 0.25, 0.35, 0.5]}


## Train YOLO

This block is intentionally minimal and kept as orchestration. The actual model training call is delegated to Ultralytics in the installed environment.

In [7]:
# Example only: adapt to the exact Ultralytics API available in the environment.
model = YOLO(EXPERIMENT['model_name'])
model.train(
    data=str(dataset_info['dataset_root'] / 'dataset.yaml'),
    imgsz=EXPERIMENT['imgsz'],
    epochs=EXPERIMENT['epochs'],
    batch=EXPERIMENT['batch'],
    pretrained=EXPERIMENT['pretrained'],
    seed=EXPERIMENT['seed'],
    project=str(experiment_root / 'runs'),
    name='yolo26n_seg',
)
print('Training step is intentionally kept for the notebook run environment.')

Ultralytics 8.4.161  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Daro\Desktop\KAGGLE\kaggle_solar_filament_segmentation\artifacts\experiments\yolo_instance_segmentation\yolo_segmentation\dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,

## Evaluation logic

The repo already uses IoU-based comparison, but the secure approach for PQ is to perform a dedicated 1:1 matching step and compute SQ/RQ/PQ explicitly. This is the same conceptual separation the project already uses in the U-Net workflow, just made explicit for YOLO.

In [10]:
# YOLO predictions on local test set

test_images_dir = dataset_info["test"] / "images"

results = model.predict(
    source=str(test_images_dir),
    imgsz=1024,
    conf=0.001,          # bajo para poder estudiar después distintos thresholds
    retina_masks=True,   # máscaras a resolución original
    save=False,
    verbose=False,
)

predictions_by_filename = {}

for result in results:
    filename = Path(result.path).name

    if result.masks is None:
        predictions_by_filename[filename] = {
            "masks": [],
            "confidences": np.array([]),
        }
        continue

    predictions_by_filename[filename] = {
        "masks": result.masks.data.cpu().numpy().astype(bool),
        "confidences": result.boxes.conf.cpu().numpy(),
    }

print(f"Images predicted: {len(predictions_by_filename)}")
print(
    f"Total predicted instances: "
    f"{sum(len(x['masks']) for x in predictions_by_filename.values())}"
)

Images predicted: 71
Total predicted instances: 12212


In [12]:
# Ground-truth instances for local test set

from src.utils.data import load_annotations, remove_duplicate_annotations
from src.processing.masks import build_masks

annotations_path = (
    raw_data_dir
    / "train"
    / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

annotations = load_annotations(annotations_path)
annotations = remove_duplicate_annotations(annotations)

# Build individual instance masks for all images
gt_masks_by_filename = build_masks(annotations)

# Keep only the local test images
test_filenames = {
    path.name
    for path in test_images_dir.iterdir()
    if path.is_file()
}

gt_instances_by_filename = {
    filename: gt_masks_by_filename[filename]
    for filename in test_filenames
    if filename in gt_masks_by_filename
}

print(f"Test images: {len(test_filenames)}")
print(f"GT images found: {len(gt_instances_by_filename)}")
print(
    f"Total GT instances: "
    f"{sum(len(instances) for instances in gt_instances_by_filename.values())}"
)

Test images: 71
GT images found: 71
Total GT instances: 494


In [ ]:
# Evaluate YOLO predictions on all test images
from src.experimental.yolo.metrics import compute_instance_pq

all_results = []

for filename, prediction in predictions_by_filename.items():

    predicted_instances = prediction["masks"]
    confidences = prediction["confidences"]

    # Apply confidence threshold
    keep = confidences >= 0.25
    predicted_instances = predicted_instances[keep]

    gt_instances = gt_instances_by_filename[filename]

    result = compute_instance_pq(
        predicted_instances,
        gt_instances,
        iou_threshold=0.5,
    )

    all_results.append({
        "filename": filename,
        **result,
    })

results_df = pd.DataFrame(all_results)

print(results_df)

print("\nMean metrics:")
print(results_df[["TP", "FP", "FN", "SQ", "RQ", "PQ"]].mean())

                 filename  TP  FP  FN        SQ        RQ        PQ
0   20110220082634Lh.jpeg   2   1   3  0.676354  0.500000  0.338177
1   20110326170914Mh.jpeg   1   0   3  0.678803  0.400000  0.271521
2   20110530122834Ch.jpeg   5   2   5  0.671853  0.588235  0.395208
3   20110615133454Bh.jpeg   5   2   3  0.734221  0.666667  0.489481
4   20110619063134Lh.jpeg   7   4   2  0.671181  0.700000  0.469827
..                    ...  ..  ..  ..       ...       ...       ...
66  20190927165910Mh.jpeg   0   1   1  0.000000  0.000000  0.000000
67  20210329015250Uh.jpeg   2   0   0  0.641106  1.000000  0.641106
68  20210731074710Th.jpeg   0   3   2  0.000000  0.000000  0.000000
69  20210817165030Mh.jpeg   2   1   0  0.645725  0.800000  0.516580
70  20220122085210Th.jpeg   2   1   0  0.674322  0.800000  0.539458

[71 rows x 7 columns]

Mean metrics:
TP    4.084507
FP    3.225352
FN    2.873239
SQ    0.589256
RQ    0.548958
PQ    0.353576
dtype: float64


## Compare to U-Net baseline

This section summarises the final comparison against the existing U-Net + connected components baseline. The notebook should report validation and local test PQ side by side.

In [15]:
baseline = {
    "val_pq": 0.3266,
    "local_test_pq": 0.2924,
}

print("U-Net baseline validation PQ:", baseline["val_pq"])
print("U-Net baseline local test PQ:", baseline["local_test_pq"])

# YOLO local test metrics
yolo_test_metrics = {
    "PQ": results_df["PQ"].mean(),
    "SQ": results_df["SQ"].mean(),
    "RQ": results_df["RQ"].mean(),
    "TP": results_df["TP"].sum(),
    "FP": results_df["FP"].sum(),
    "FN": results_df["FN"].sum(),
}

comparison_df = pd.DataFrame(
    [
        {
            "model": "U-Net",
            "split": "local_test",
            "PQ": baseline["local_test_pq"],
            "SQ": None,
            "RQ": None,
            "TP": None,
            "FP": None,
            "FN": None,
        },
        {
            "model": "YOLO",
            "split": "local_test",
            **yolo_test_metrics,
        },
    ]
)

print(comparison_df)

U-Net baseline validation PQ: 0.3266
U-Net baseline local test PQ: 0.2924
   model       split      PQ  SQ  RQ   TP   FP   FN
0  U-Net  local_test  0.2924 NaN NaN  NaN  NaN  NaN
1   YOLO  local_test     NaN NaN NaN  0.0  0.0  0.0


## Prediction visualisations

Display a few representative validation and local test examples, with original image, GT instances, and YOLO predictions overlaid.

In [ ]:
# Example placeholder for later integration with real YOLO outputs.
print('Visualisation block ready for actual predictions.')